In [1]:
import pandas as pd
import numpy as np

# Считываем первые 1 000 000 строк таблицы транзакций
transactions = pd.read_csv('/content/transactions.csv', nrows=1000000)

# Считываем остальные таблицы
gender_train = pd.read_csv('/content/gender_train.csv')
tr_mcc_codes = pd.read_csv('/content/tr_mcc_codes.csv', sep=';')
tr_types = pd.read_csv('/content/tr_types.csv', sep=';')

In [2]:
# ЗАДАНИЕ 1
# 1. Выбираем случайные 1000 строк (точнее, только столбец tr_type)
# random_state=42 зафиксирует результат, чтобы он не менялся при перезапуске
sample_tr_types = transactions['tr_type'].sample(n=1000, random_state=42)

# 2. Находим в tr_types те tr_type, в описании которых есть 'POS' или 'ATM'
# na=False нужен на случай, если в описании есть пропуски (NaN)
matched_types = tr_types[tr_types['tr_description'].str.contains('POS|ATM', case=True, na=False)]['tr_type']

# Вычисляем долю: проверяем, сколько элементов из выборки входят в список matched_types
share = sample_tr_types.isin(matched_types).mean()
print(f"Доля транзакций с 'POS' или 'ATM': {share}")

Доля транзакций с 'POS' или 'ATM': 0.384


In [3]:
# ЗАДАНИЕ 2
# 1. Считаем частоту встречаемости каждого tr_type
top_10_frequencies = transactions['tr_type'].value_counts().head(10)

# 2. Выводим топ-10 вместе с описанием без использования merge
# Превращаем результат в DataFrame для красивого вывода
top_10_df = top_10_frequencies.reset_index()
top_10_df.columns = ['tr_type', 'count']

# Добавляем текстовое описание из таблицы tr_types через метод map
# (он сопоставляет tr_type с tr_description как словарь)
mapping_dict = tr_types.set_index('tr_type')['tr_description']
top_10_df['tr_description'] = top_10_df['tr_type'].map(mapping_dict)

print("Топ-10 транзакций по частоте:")
print(top_10_df)

Топ-10 транзакций по частоте:
   tr_type   count                                     tr_description
0   1010.0  118572                              Покупка. POS ТУ СБ РФ
1   7070.0   99746  Перевод на карту (с карты) через Мобильный бан...
2   2010.0   79414              Выдача наличных в АТМ Сбербанк России
3   1110.0   76702                             Покупка. POS ТУ Россия
4   1030.0   60474                     Оплата услуги. Банкоматы СБ РФ
5   2370.0   27187  Списание с карты на карту по операции <перевод...
6   7030.0   17291  Перевод на карту (с карты) через АТМ (в предел...
7   7010.0   14539       Взнос наличных через АТМ (в своем тер.банке)
8   1100.0    8532                                Покупка. ТУ  Россия
9   1200.0    6134                               Покупка. Зарубеж. ТУ


In [4]:
# ЗАДАНИЕ 3.
# 1. Клиент с максимальной суммой приходов (amount > 0)
income_by_customer = transactions[transactions['amount'] > 0].groupby('customer_id')['amount'].sum()
best_income_customer = income_by_customer.idxmax()
max_income_value = income_by_customer.max()

# 2. Клиент с максимальной суммой расходов (amount < 0)
# Суммируем отрицательные значения. Тот, у кого сумма самая минимальная (наибольшая по модулю), потратил больше всех.
expense_by_customer = transactions[transactions['amount'] < 0].groupby('customer_id')['amount'].sum()
best_expense_customer = expense_by_customer.idxmin()
max_expense_value = expense_by_customer.min()

print(f"ID клиента с макс. приходом: {best_income_customer} (Сумма: {max_income_value})")
print(f"ID клиента с макс. расходом: {best_expense_customer} (Сумма: {max_expense_value})")

# 3. Модуль разницы между расходами и приходами ДЛЯ ЭТИХ ДВУХ КЛИЕНТОВ.
# Посчитаем общие расходы и общие приходы для каждого из них.
def get_totals(customer_id):
    customer_data = transactions[transactions['customer_id'] == customer_id]
    pos_sum = customer_data[customer_data['amount'] > 0]['amount'].sum()
    neg_sum = customer_data[customer_data['amount'] < 0]['amount'].sum()
    return pos_sum, neg_sum

# Данные для "богатого" клиента
inc_cust_pos, inc_cust_neg = get_totals(best_income_customer)
# Данные для "тратящего" клиента
exp_cust_pos, exp_cust_neg = get_totals(best_expense_customer)

# Разница для клиента с макс. приходом
diff_inc_cust = abs(inc_cust_neg - inc_cust_pos)
# Разница для клиента с макс. расходом
diff_exp_cust = abs(exp_cust_neg - exp_cust_pos)

print(f"Модуль разницы расходов и приходов для клиента {best_income_customer}: {diff_inc_cust}")
print(f"Модуль разницы расходов и приходов для клиента {best_expense_customer}: {diff_exp_cust}")

ID клиента с макс. приходом: 70780820 (Сумма: 1248114886.81)
ID клиента с макс. расходом: 70780820 (Сумма: -1249952204.79)
Модуль разницы расходов и приходов для клиента 70780820: 2498067091.6
Модуль разницы расходов и приходов для клиента 70780820: 2498067091.6


In [5]:
# ЗАДАНИЕ 4
# 1. Метрики по топ-10 типам транзакций
top_10_types_list = top_10_frequencies.index.tolist()
top_10_stats = transactions[transactions['tr_type'].isin(top_10_types_list)].groupby('tr_type')['amount'].agg(['mean', 'median'])

print("Среднее и медиана по топ-10 типам транзакций:")
print(top_10_stats)

# 2. Метрики для двух клиентов из Задания 3
target_customers = [best_income_customer, best_income_customer] # ID из предыдущего шага
customers_stats = transactions[transactions['customer_id'].isin(target_customers)].groupby('customer_id')['amount'].agg(['mean', 'median'])

print("\nСреднее и медиана по суммам для выбранных клиентов:")
print(customers_stats)

Среднее и медиана по топ-10 типам транзакций:
                  mean     median
tr_type                          
1010.0   -22125.767791   -7995.46
1030.0    -5944.288715   -2245.92
1100.0   -49451.949674  -11341.87
1110.0   -34231.202008  -11566.47
1200.0   -39466.308228   -7389.06
2010.0  -143884.212043  -44918.32
2370.0  -237293.409044  -53901.98
7010.0   317795.220691  112295.79
7030.0    73235.371657   11678.76
7070.0    59917.740957   10443.51

Среднее и медиана по суммам для выбранных клиентов:
                  mean   median
customer_id                    
70780820    -20.694946  8803.99
